---
## Lista de funciones avanzada

La forma directa es generar una lista y pasar las funciones al metodo `.agg()`


In [ ]:
import pandas as pd

df = pd.DataFrame({
    "categoria": ["Port\u00e1tiles","Port\u00e1tiles","Port\u00e1tiles","Monitores","Monitores","Perif\u00e9ricos","Perif\u00e9ricos","Perif\u00e9ricos","Perif\u00e9ricos"],
    "importe":   [899, 899, 799, 349, 349, 45, 25, 45, 35],
    "unidades":  [1, 2, 1, 1, 3, 5, 2, 4, 1]
})
# Cada funcion equivale a una columna, cada categoría tiene su resultado calculado
lista_funciones = df.groupby("categoria")["importe"].agg(["mean", "sum", "count", "min", "max"])
print('Una columna por función: ')
print(lista_funciones)

---
# **.agg()** con diccionario

Funciones distintas por columna, cuando se quieren aplicar funciones diferentes a columnas diversas, se pasa un diccionario donde las claves son los nombres de columna y los valores son las funciones,

In [ ]:
resultado = df.groupby("categoria").agg({ "importe":  ["mean", "sum"],
    "unidades": ["sum", "max"]})
print(resultado)

---
# El resultado tiene un **Multindex** en las columnas

El primer nivel es el nombre de la columna original; la segunda de la función aplicada.
Esto puede dificultar el acceso posterior a las columnas. La solución es aplanar el `MultiIndex`.

In [ ]:
#El multindez tiene un resultado real solamente la primera vez que se ejecuta...

#resultado.columns = ['_'.join(col) for col in resultado.columns]
#resultado = resultado.reset_index()
#print(resultado)

---
# Agregaciones nombradas - **forma más limpia**

La forma mas profesional y legible de usar `.agg()` es con agregaciones nombradas. Se pasan tuplas donde el primer elemento es el nombre deseado para la columna resultado y el segundo es la función.

In [ ]:
resultado = df.groupby("categoria")["importe"].agg(
    importe_medio=("mean"),
    renueve_total=("sum"),
    numero_ventas=("count"),
    precio_minimo=("min"),
    precio_maximo=("max")
).reset_index().sort_values("renueve_total", ascending=False)
print(resultado)

---
# Groupby por dos columnas con múltiples métricas

La combinación de groupby por dos columnas con el `.agg()` produce el análisis mas completo. Cada combinación única de valores en ambas columnas forma un grupo independiente.

In [ ]:
df["ciudad"] = ["Madrid","Madrid","Barcelona","Madrid","Barcelona","Madrid","Sevilla","Barcelona","Sevilla"]

resultado = (
    df
    .groupby(["categoria", "ciudad"])
    .agg(
        num_ventas    = ("importe", "count"),
        revenue_total = ("importe", "sum"),
        ticket_medio  = ("importe", "mean")
    )
    .reset_index()
    .sort_values(["categoria", "revenue_total"], ascending=[True, False])
)

print(resultado)

---
# **transform()**: agregar sin reducir filas

Variante del groupby que no reduce el número de filas. `transform()` calcula el valor agregado del grupo pero lo repite en cada fila que pertenece a ese grupo, manteniendo el mismo shape que el df. original.

In [ ]:
# Añadir el revenue total de la categoría como columna en cada fila
df["revenue_categoria"] = df.groupby("categoria")["importe"].transform("sum")

# Calcular la contribución de cada venta sobre el total de su categoría
df["pct_sobre_categoria"] = (df["importe"] / df["revenue_categoria"] * 100).round(1)

print(df[["categoria", "importe", "revenue_categoria", "pct_sobre_categoria"]])

## `size()` vs `count()`: cuándo usar cada uno

Ambos cuentan filas por grupo pero se comportan diferente ante los nulos.

`size()` devuelve el número total de filas en cada grupo, incluyendo las que tienen nulos en cualquier columna.

`count()` devuelve el número de valores no nulos por columna y por grupo. Si una columna tiene nulos, su conteo será menor que el total de filas del grupo.

In [ ]:
df_nulos = df.copy()
df_nulos.loc[0, "importe"] = None  # introducir un nulo

print(df_nulos.groupby("categoria").size())
print("---")
print(df_nulos.groupby("categoria")["importe"].count())

---
# `filter()`: conservar o eliminar grupos completos

`filter()` evalúa una condición sobre cada grupo y decide si incluir o descartar **todas sus filas**. No transforma ni agrega — es un filtro todo-o-nada a nivel de grupo.

A diferencia de un filtro fila a fila con `.query()` o corchetes, `filter()` permite condiciones que dependen del comportamiento del grupo completo, como mantener solo las categorías cuyo revenue total supera un umbral.

In [ ]:
# filter() recibe una función lambda que se aplica a cada grupo completo
# g representa el grupo como DataFrame — aquí se accede a la columna 'importe' del grupo y se suma
# si la suma supera 500 la función devuelve True → el grupo se conserva con todas sus filas
# si devuelve False → todas las filas del grupo desaparecen del resultado
resultado_filtrado = df.groupby("categoria").filter(lambda g: g["importe"].sum() > 500)

print("Categorías con revenue total > 500:")
print(resultado_filtrado[["categoria", "importe"]])

# ~ invierte la condición booleana: selecciona filas que NO están en resultado_filtrado
# .unique() extrae los valores únicos de la columna para pasarlos a .isin()
descartadas = df[~df["categoria"].isin(resultado_filtrado["categoria"].unique())]
print("\nCategorías descartadas (revenue <= 500):")
print(descartadas[["categoria", "importe"]])

---
# `apply()`: lógica personalizada por grupo

`apply()` es la herramienta más flexible del groupby. Pasa cada grupo como un DataFrame independiente a una función — lo que permite aplicar cualquier operación que `agg()` o `transform()` no puedan expresar.

El resultado puede ser un escalar, una Serie o un DataFrame completo. Esa flexibilidad lo hace útil cuando la lógica de negocio no encaja en ninguna función estándar de pandas — por ejemplo, seleccionar las N filas con mayor valor dentro de cada grupo.

In [ ]:
# se define una función normal que recibirá cada grupo como DataFrame independiente
def top_ventas(grupo):
    # grupo es un DataFrame con solo las filas de esa categoría
    # .nlargest(2, "importe") devuelve las 2 filas con mayor valor en la columna importe
    return grupo.nlargest(2, "importe")

# group_keys=False evita que pandas añada el nombre del grupo como nivel extra en el índice del resultado
resultado = df.groupby("categoria", group_keys=False).apply(top_ventas)

print("Top 2 ventas por categoría:")
print(resultado[["categoria", "ciudad", "importe"]])

---
# `rank()` dentro de grupos

`rank()` asigna una posición ordinal a cada fila dentro de su grupo. El resultado es una columna nueva que indica el puesto de cada fila según la métrica elegida, sin reducir filas ni agregar.

Se comporta como `transform()` — mantiene el shape original — pero en vez de propagar un agregado, asigna una posición relativa. Útil para construir rankings por segmento directamente sobre el DataFrame.

In [ ]:
# groupby("categoria") agrupa; .rank() asigna posición dentro de cada grupo por separado
# method="dense" evita saltos en la numeración ante empates:
#   si dos filas empatan en posición 1, la siguiente es 2, no 3
# ascending=False → posición 1 = mayor importe (más caro = mejor puesto)
df["ranking_categoria"] = (
    df.groupby("categoria")["importe"]
    .rank(method="dense", ascending=False)
)

# sort_values con lista de dos columnas: ordena primero por categoria y luego por ranking
print(
    df[["categoria", "importe", "ranking_categoria"]]
    .sort_values(["categoria", "ranking_categoria"])
)

---
# Suma acumulada por grupo con `cumsum()`

`cumsum()` calcula la suma acumulada dentro de cada grupo. Cada fila muestra el total acumulado desde la primera fila del grupo hasta la actual, y la acumulación se reinicia al comenzar un nuevo grupo.

Útil para ver cómo crece el revenue dentro de cada categoría de forma progresiva — análisis de progresión o curvas de crecimiento por segmento sin necesidad de crear un DataFrame separado.

In [ ]:
# .cumsum() tras groupby reinicia la acumulación al cambiar de grupo
# la primera fila de cada categoría muestra su propio importe
# la siguiente muestra primera + segunda, y así sucesivamente
# al cambiar de categoría el contador vuelve a cero
df["acumulado_categoria"] = df.groupby("categoria")["importe"].cumsum()

# sort_values por categoria para que los grupos aparezcan juntos y se lea bien la acumulación
print(
    df[["categoria", "importe", "acumulado_categoria"]]
    .sort_values("categoria")
)